# Phase 4: Feature Engineering (Clinical Scoring)
In this notebook, we take the cleaned numeric dataframe from Phase 1 and apply the clinical scoring rules from the official Codebooks to create our predictive targets.

In [ ]:
# Load the CSV exported from Phase 1 Data Cleaning
try:
    import pandas as pd
    import numpy as np
    
    df_clean = pd.read_csv('data/Dataset_Cleaned_Imputed.csv')
    print(f'Clean data loaded successfully! Shape: {df_clean.shape}')
except FileNotFoundError:
    print('Error: Dataset_Cleaned_Imputed.csv not found. Please run p1_data_cleaning.ipynb first!')

Clean data loaded successfully! Shape: (410, 57)


### 1. EPDS (Postpartum Depression)
The EPDS has 10 questions. Questions 3, 5, 6, 7, 8, 9, 10 must be reverse-scored (0->3, 1->2, 2->1, 3->0) before summing.
A total score >= 11 indicates high PPD Risk.

In [7]:
def score_epds(df):
    df_feat = df.copy()
    epds_cols = [f'epds_{i}' for i in range(1, 11)]
    reverse_items = ['epds_3', 'epds_5', 'epds_6', 'epds_7', 'epds_8', 'epds_9', 'epds_10']
    
    # Apply reverse scoring (3 - value)
    for item in reverse_items:
        if item in df_feat.columns:
            df_feat[item] = 3 - df_feat[item]
        
    # Calculate totals
    df_feat['epds_total'] = df_feat[epds_cols].sum(axis=1)
    
    # Create Binary Target
    df_feat['ppd_risk'] = (df_feat['epds_total'] >= 11).astype(int)
    return df_feat

### 2. HADS-A (Anxiety Subscale)
The dataset contains odd-numbered HADS questions (1, 3, 5, 7, 9, 11, 13). According to the HADS PDF, these constitute the Anxiety subscale.
Scores: 0-7 = Normal, 8-10 = Borderline, 11-21 = Abnormal.

In [8]:
def score_hads(df):
    df_feat = df.copy()
    hads_cols = [f'hads_{i}' for i in [1, 3, 5, 7, 9, 11, 13]]
    
    df_feat['hads_a_total'] = df_feat[hads_cols].sum(axis=1)
    
    # Categorize
    conditions = [
        (df_feat['hads_a_total'] <= 7),
        (df_feat['hads_a_total'] >= 8) & (df_feat['hads_a_total'] <= 10),
        (df_feat['hads_a_total'] >= 11)
    ]
    choices = ['Normal', 'Borderline', 'Abnormal']
    df_feat['anxiety_risk'] = np.select(conditions, choices, default='Unknown')
    return df_feat

### 3. CBTS (PTSD/Birth Trauma)
The CBTS requires calculating two subscales: Birth-Related PTSD (Q3-Q12) and General PTSD (Q13-Q22).

In [9]:
def score_cbts(df):
    df_feat = df.copy()
    
    # Exact dataset columns: cbts_m_3 to cbts_m_12 and cbts_13 to cbts_22
    birth_ptsd_cols = [f'cbts_m_{i}' for i in range(3, 13)]
    general_ptsd_cols = [f'cbts_{i}' for i in range(13, 23)]
    
    df_feat['cbts_birth_ptsd'] = df_feat[birth_ptsd_cols].sum(axis=1)
    df_feat['cbts_general_ptsd'] = df_feat[general_ptsd_cols].sum(axis=1)
    df_feat['cbts_total'] = df_feat['cbts_birth_ptsd'] + df_feat['cbts_general_ptsd']
    return df_feat

### 4. Apply all Feature Engineering Functions

In [10]:
# Execute the scoring logic
df_features = score_epds(df_clean)
df_features = score_hads(df_features)
df_features = score_cbts(df_features)

print("Feature Engineering Complete.")
df_features[['epds_total', 'ppd_risk', 'hads_a_total', 'anxiety_risk', 'cbts_total']].head()

Feature Engineering Complete.


,epds_total,ppd_risk,hads_a_total,anxiety_risk,cbts_total
0,16,1,9,Borderline,6
1,21,1,3,Normal,0
2,17,1,9,Borderline,9
3,13,1,13,Abnormal,14
4,18,1,3,Normal,3


In [12]:
df_features.head()

,age,marital_status,education,gestationnal_age,type_pregnancy,sex,cbts_m_3,cbts_m_4,cbts_m_5,cbts_m_6,...,sleep_night_duration_bb1,night_awakening_number_bb1,how_falling_asleep_bb1,epds_total,ppd_risk,hads_a_total,anxiety_risk,cbts_birth_ptsd,cbts_general_ptsd,cbts_total
0,34,in a relationship,university,37.0,single,girl,0,0,0,0,...,10.0,3,while being rocked,16,1,9,Borderline,1,5,6
1,33,in a relationship,university,42.0,single,boy,0,0,0,0,...,11.0,0,alone in the crib,21,1,3,Normal,0,0,0
2,37,in a relationship,university,41.0,single,girl,0,0,1,0,...,12.0,1,while being rocked,17,1,9,Borderline,2,7,9
3,31,in a relationship,university,37.5,single,boy,0,0,1,1,...,11.0,2,while being fed,13,1,13,Abnormal,6,8,14
4,36,single,university,40.0,single,boy,0,0,0,0,...,10.5,1,alone in the crib,18,1,3,Normal,0,3,3


In [13]:
# Final Export for Machine Learning (Phases 5 & 6)
df_features.to_csv('Dataset_Featured.csv', index=False)
print('Engineered dataset successfully exported to Dataset_Featured.csv!')

Engineered dataset successfully exported to Dataset_Featured.csv!
